In [8]:
import sys
from pathlib import Path
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [11]:
ds_sample = ds["train"].shuffle(seed=42).select(range(100))

In [12]:
from langchain_core.documents import Document
import uuid

In [13]:
all_documents = []

for doc_idx, sample in enumerate(ds_sample):
    raws = sample['context']['sentences']
    # Attach elements in same array (sentences), add linebreak between arrays (paragraphs)
    full_doc_content = "\n\n".join(["".join(paragraph) for paragraph in raws])
    
    for chunk_idx, paragraph_sentences in enumerate(raws):
        # Attach elements in same array (sentences)
        chunk_content = "".join(paragraph_sentences)
        
        chunk_id = f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}"
        doc = Document(
            id=str(uuid.uuid4()), 
            page_content=chunk_content,
            metadata={
                "doc_idx": doc_idx + 1,
                "chunk_id": f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}",
                "doc_content": full_doc_content,
                "title": sample['context']['title'][chunk_idx],
                "question": sample['question'],
                "answer": sample['answer']
            }
        )
        all_documents.append(doc)

print(f"Created {len(all_documents)} documents from {len(ds_sample)} samples.")

Created 990 documents from 100 samples.


In [14]:
all_documents[0]

Document(id='955b32e8-7e1e-45e1-bb26-188a96ae9f76', metadata={'doc_idx': 1, 'chunk_id': 'doc_1_chunk_1', 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The po

In [15]:
# Inspect the first document
all_documents[0].metadata['doc_content']

'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the 2010 census. North Haven is accessed by three-times daily ferry service from Rockland, or by air

In [16]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

In [17]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""
 
CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>
 
Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""
 
 
def situate_context(chunks: list[Document]) -> dict[str, str]:
    prompts = []
    for chunk in chunks:
        prompt = [
            {"role": "system", "content": "You MUST answer in Korean."},
                {"role": "user", "content": DOCUMENT_CONTEXT_PROMPT.format(doc_content=chunk.metadata["doc_content"])},
                {"role": "user", "content": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk.page_content)},
            ]
        prompts.append(prompt)
    response = llm.batch(prompts)
    return response

In [18]:
res = situate_context(all_documents)

In [19]:
for r, chunk in zip(res, all_documents):
    chunk.page_content =  r.content + "\n\n" + chunk.page_content
    chunk.metadata["contextualized_content"] = r.content
    chunk.metadata["original_content"] = chunk.page_content   

In [20]:
all_documents[0].metadata

{'doc_idx': 1,
 'chunk_id': 'doc_1_chunk_1',
 'doc_content': 'Vinalhaven is a town located on the larger of the two Fox Islands in Knox County, Maine, United States. Vinalhaven is also used to refer to the Island itself. The population was 1,165 at the 2010 census. It is home to a thriving lobster fishery and hosts a summer colony. Since there is no bridge to the island, Vinalhaven is accessible from Rockland via an approximately hour-and-fifteen-minute ferry ride across West Penobscot Bay, or by air taxi from Knox County Regional Airport.\n\nOwls Head is a town in Knox County, Maine, United States. The population was 1,580 at the 2010 census. A resort and fishing area, the community is home to the Knox County Regional Airport. It includes the village of Ash Point.\n\nNorth Haven is a town in Knox County, Maine, United States, in Penobscot Bay. The town is both a year-round island community and a prominent summer colony. The population was 355 at the 2010 census. North Haven is accesse

In [21]:
i = 0
for doc in all_documents:
    if 'original_content' not in doc.metadata:
        print(doc.id)
        print(doc.metadata['doc_idx'])
        print(doc.page_content)
        print("-"*100)
        i += 1
i

0

In [22]:
import pickle

In [23]:
with open(root / "datasets" / "hotpotqa.pkl", "wb") as f:
    pickle.dump(all_documents, f)

In [24]:
# Create evaluation dataset
# Golden chunks are identified by matching supporting_facts titles with context titles

eval_dataset = []

for doc_idx, sample in enumerate(ds_sample):
    # Get the titles of supporting facts (golden paragraphs)
    golden_titles = set(sample['supporting_facts']['title'])
    context_titles = sample['context']['title']
    
    # Find chunk indices where title matches a supporting fact title
    golden_chunk_ids = []
    for chunk_idx, title in enumerate(context_titles):
        if title in golden_titles:
            chunk_id = f"doc_{doc_idx + 1}_chunk_{chunk_idx + 1}"
            golden_chunk_ids.append(chunk_id)
    
    eval_entry = {
        "query": sample['question'],
        "answer": sample['answer'],
        "golden_chunk_ids": golden_chunk_ids,
        "doc_idx": doc_idx + 1
    }
    eval_dataset.append(eval_entry)

print(f"Created eval dataset with {len(eval_dataset)} entries.")

Created eval dataset with 100 entries.


In [25]:
# Save eval dataset
import json

with open(root / "datasets" / "hotpotqa_eval.json", "w") as f:
    json.dump(eval_dataset, f, indent=2)
    
print("Saved eval dataset to datasets/hotpotqa_eval.json")

Saved eval dataset to datasets/hotpotqa_eval.json


In [26]:
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="embedding-passage")
vectorstore = FAISS.from_documents(documents=all_documents, embedding=embeddings)
vectorstore.save_local(root / "faiss_index", "hotpotqa_100")